# 🧠 BSPC (Elsevier Q1) Production Experiments Launcher
### Ferdowsi Cloud / GPU Server Execution Suite

This notebook provides a 1-click execution pipeline for the 5-fold cross-validation experiments on Ferdowsi Cloud (JupyterLab).

* **Model:** Attention Res-UNet (GroupNorm, Deep Supervision, Stochastic Depth)
* **Loss:** Compound Focal Tversky ($\alpha=0.8, \beta=0.2, \gamma=0.6$) + Binary Focal Loss
* **Protocols:**
  1. `patient_grouped`: 5-Fold GroupKFold isolating 110 patients (Zero Leakage)
  2. `random_slice`: Matched 5-Fold KFold across slices (Reviewer 3 Leakage Quantification)
* **Speed:** Uses precomputed in-memory cache (`preprocessed_cache_128.npz`) for **100% GPU compute efficiency** (zero I/O wait).

## 📦 Step 1: Verify Offline Environment & Libraries


In [ ]:
# Server is offline (No internet/PyPI). All modules run on pure TensorFlow, NumPy, SciPy!
import sys
import numpy as np
import tensorflow as tf
import scipy
import pandas as pd

print('=' * 65)
print('TensorFlow:', tf.__version__, 'NumPy:', np.__version__, 'SciPy:', scipy.__version__)
print('✅ Pure offline execution mode active: ZERO cv2/skimage dependencies!')
print('=' * 65)


## ⚡ Step 2: Verify GPU Hardware & Enable Memory Growth

In [ ]:
import tensorflow as tf
import os

print("=" * 65)
print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        print(f"✅ GPU Detected: {gpu}")
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
            print(f"   --> Dynamic memory growth enabled for {gpu.name}")
        except Exception as e:
            print(f"   --> Note: {e}")
else:
    print("⚠️ WARNING: No GPU detected! Please ensure a GPU instance is allocated.")
print("=" * 65)
!nvidia-smi

## 🔍 Step 3: Verify Preprocessed Cache & Screening Audit
* Checks `kaggle_3m/preprocessed_cache_128.npz` (85.8 MB).
* Loads in 0.5s if pre-uploaded, or builds it automatically if missing.

In [ ]:
import sys
import os
if "codes" not in sys.path:
    sys.path.insert(0, "codes")

from src.config import CFG
from src.dataset import build_dataframe
from src.preprocessing import get_or_create_preprocessed_cache

data_root = "kaggle_3m" if os.path.exists("kaggle_3m") else CFG["DATA_ROOT"]
data_csv = os.path.join(data_root, "data.csv")
output_dir = os.path.join("codes", "results", "bspc_production")
cache_path = os.path.join(data_root, "preprocessed_cache_128.npz")
audit_csv = os.path.join(output_dir, "cohort_screening_audit.csv")

os.makedirs(output_dir, exist_ok=True)
df = build_dataframe(data_root=data_root, data_csv_path=data_csv)

imgs, msks, df = get_or_create_preprocessed_cache(
    df=df,
    cache_path=cache_path,
    audit_csv_path=audit_csv
)
print(f"\n✅ Ready: {len(df)} slices in RAM ({imgs.nbytes / (1024*1024):.1f} MB)")

## 🚀 Step 4: Run Main 5-Fold Patient-Grouped Cross-Validation (Zero Patient Leakage)
* Trains 5 folds using `GroupKFold` on 110 patients with internal 15% calibration split.
* Filters 200 artifact slices from training, retains all slices in validation/test.
* Exports raw float32 logits, probs, targets, and `is_artifact_flagged` into `.npz` files.

In [ ]:
!python codes/scripts/run_gpu_experiments.py \
    --data_root kaggle_3m \
    --data_csv kaggle_3m/data.csv \
    --output_dir codes/results/bspc_production \
    --cache_path kaggle_3m/preprocessed_cache_128.npz \
    --split_type patient_grouped \
    --epochs 100 \
    --batch_size 16 \
    --base_filters 32 \
    --patience 20 \
    --resume

## 🧪 Step 5: Run Matched 5-Fold Random-Slice CV (Leakage Quantification for Reviewer 3)
* Matched 5-fold slice split proving artificial performance inflation and calibration degradation.
* Saved in isolated `codes/results/bspc_production/random_slice/` folder (zero conflict).

In [ ]:
!python codes/scripts/run_gpu_experiments.py \
    --data_root kaggle_3m \
    --data_csv kaggle_3m/data.csv \
    --output_dir codes/results/bspc_production \
    --cache_path kaggle_3m/preprocessed_cache_128.npz \
    --split_type random_slice \
    --epochs 100 \
    --batch_size 16 \
    --base_filters 32 \
    --patience 20 \
    --resume

## 📦 Step 6: Package & Zip All Results for 1-Click Download
Creates `bspc_results_bundle.zip` containing all fold weights, `.npz` prediction maps, and history logs.

In [ ]:
import shutil
import os

results_dir = os.path.join("codes", "results", "bspc_production")
zip_output = "bspc_results_bundle"

print(f"Compressing '{results_dir}' into zip archive...")
shutil.make_archive(zip_output, "zip", results_dir)

zip_filename = f"{zip_output}.zip"
if os.path.exists(zip_filename):
    size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
    print(f"\n✅ SUCCESS! Download archive created: {zip_filename} ({size_mb:.2f} MB)")
    print(">>> In JupyterLab sidebar, right-click 'bspc_results_bundle.zip' and select 'Download'!")
else:
    print("Error creating zip archive.")